Read and understand the data

In [2]:
import pandas as pd

flights_raw = pd.read_csv('data/flights.csv')
airports_raw = pd.read_csv('data/airports.csv')


flights_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   date         20000 non-null  str  
 1   delay        20000 non-null  int64
 2   distance     20000 non-null  int64
 3   origin       20000 non-null  str  
 4   destination  20000 non-null  str  
dtypes: int64(2), str(3)
memory usage: 781.4 KB


In [21]:
flights_raw.head(10)

,date,delay,distance,origin,destination
0,2001-01-01 00:47:00,66,1750,DTW,LAS
1,2001-01-01 01:10:00,95,2399,HNL,SFO
2,2001-01-01 01:24:00,-5,407,LAS,OAK
3,2001-01-01 01:39:00,4,256,LAS,PHX
4,2001-01-01 06:02:00,-6,377,MHT,BWI
5,2001-01-01 06:05:00,-27,370,MDT,DTW
6,2001-01-01 06:17:00,-7,813,AUS,ATL
7,2001-01-01 06:22:00,-26,931,DCA,MSP
8,2001-01-01 06:30:00,29,236,LAS,LAX
9,2001-01-01 06:35:00,-4,370,BWI,BOS


In [22]:
airports_raw.head(10)

,iata,name,city,state,country,latitude,longitude
0,00M,Thigpen,Bay Springs,MS,USA,31.953765,-89.234505
1,00R,Livingston Municipal,Livingston,TX,USA,30.685861,-95.017928
2,00V,Meadow Lake,Colorado Springs,CO,USA,38.945749,-104.569893
3,01G,Perry-Warsaw,Perry,NY,USA,42.741347,-78.052081
4,01J,Hilliard Airpark,Hilliard,FL,USA,30.688012,-81.905944
5,01M,Tishomingo County,Belmont,MS,USA,34.491667,-88.201111
6,02A,Gragg-Wade,Clanton,AL,USA,32.850487,-86.611453
7,02C,Capitol,Brookfield,WI,USA,43.087510,-88.177869
8,02G,Columbiana County,East Liverpool,OH,USA,40.673313,-80.641406
9,03D,Memphis Memorial,Memphis,MO,USA,40.447259,-92.226961


A row basically shows a flight with the above columns. The relevant columns for us are: flight date, delay, distance and the origin.

In [3]:
airports_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3376 entries, 0 to 3375
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   iata       3376 non-null   str    
 1   name       3376 non-null   str    
 2   city       3364 non-null   str    
 3   state      3364 non-null   str    
 4   country    3376 non-null   str    
 5   latitude   3376 non-null   float64
 6   longitude  3376 non-null   float64
dtypes: float64(2), str(5)
memory usage: 184.8 KB


Cleaning three problems

Change the date-column while reading the data. Read again

In [9]:
flights = pd.read_csv('data/flights.csv', parse_dates=['date'])

flights.info()


<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         20000 non-null  datetime64[us]
 1   delay        20000 non-null  int64         
 2   distance     20000 non-null  int64         
 3   origin       20000 non-null  str           
 4   destination  20000 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 781.4 KB


Find empty columns, duplicate rows and also critically assessing the head() data makes one suspicious if the negative delay values are reasonable. 

In [10]:
empty = flights.columns[flights.isna().all()]

print(len(empty))
print(list(empty))



0
[]


No "totally" empty columns. 

Find if there are missing date, delay, distance and origin rows

In [ ]:
print(flights['date'].isna().sum())
print((flights['date'].isna().mean() * 100).round(1))

print(flights['delay'].isna().sum())
print((flights['delay'].isna().mean() * 100).round(1))

print(flights['distance'].isna().sum())
print((flights['distance'].isna().mean() * 100).round(1))

print(flights['origin'].isna().sum())
print((flights['origin'].isna().mean() * 100).round(1))



0
0.0
0
0.0
0
0.0
0
0.0


In [26]:
airports_raw.isna().sum()

iata          0
name          0
city         12
state        12
country       0
latitude      0
longitude     0
dtype: int64

Duplicates?

In [15]:
print(flights.duplicated().sum())

airports = airports_raw

print(airports.duplicated().sum())

0
0


Reset the neagative delays to zero or leave them for now...

In [ ]:
negative_delayed = flights[flights['delay'] < 0]

,date,delay,distance,origin,destination
2,2001-01-01 01:24:00,-5,407,LAS,OAK
4,2001-01-01 06:02:00,-6,377,MHT,BWI
5,2001-01-01 06:05:00,-27,370,MDT,DTW
6,2001-01-01 06:17:00,-7,813,AUS,ATL
7,2001-01-01 06:22:00,-26,931,DCA,MSP
...,...,...,...,...,...
19991,2001-03-31 20:15:00,-7,248,SAT,DAL
19992,2001-03-31 20:16:00,-15,36,SNA,LAX
19993,2001-03-31 20:40:00,-5,677,HOU,ELP
19994,2001-03-31 20:50:00,-4,410,SLC,COS


Changing 9720 rows to zero for delays (leave them for now)

12 missing values for city and state. Need to replace them? But we need to know with what first! Or if its better to leave them as they are. Or replace them wth some kind of place holder.

In [27]:
airports.isna().sum()

iata          0
name          0
city         12
state        12
country       0
latitude      0
longitude     0
dtype: int64

In [29]:
airports[airports['city'].isna()]

,iata,name,city,state,country,latitude,longitude
1136,CLD,MC Clellan-Palomar Airport,NaN,NaN,USA,33.127231,-117.278727
1715,HHH,Hilton Head,NaN,NaN,USA,32.224384,-80.697629
2251,MIB,Minot AFB,NaN,NaN,USA,48.415769,-101.358039
2312,MQT,Marquette County Airport,NaN,NaN,USA,46.353639,-87.395361
2752,RCA,Ellsworth AFB,NaN,NaN,USA,44.145094,-103.103567
2759,RDR,Grand Forks AFB,NaN,NaN,USA,47.961167,-97.401167
2794,ROP,Prachinburi,NaN,NaN,Thailand,14.078333,101.378334
2795,ROR,Babelthoup/Koror,NaN,NaN,Palau,7.367222,134.544167
2900,SCE,University Park,NaN,NaN,USA,40.851206,-77.846302
2964,SKA,Fairchild AFB,NaN,NaN,USA,47.615058,-117.655803


As they are so few 12/20000 then one can neglect them and keep them as they are. another alternative would have been to search each airport and find what corresponding city and state would be.

Merge with the second table

In [ ]:
flights.shape

(20000, 5)

In [ ]:
merged = pd.merge(flights, airports, left_on="origin", right_on="iata", how="left")

merged.shape

(20000, 12)

In [35]:
merged.head()

,date,delay,distance,origin,destination,iata,name,city,state,country,latitude,longitude
0,2001-01-01 00:47:00,66,1750,DTW,LAS,DTW,Detroit Metropolitan-Wayne County,Detroit,MI,USA,42.212059,-83.348836
1,2001-01-01 01:10:00,95,2399,HNL,SFO,HNL,Honolulu International,Honolulu,HI,USA,21.318691,-157.922407
2,2001-01-01 01:24:00,-5,407,LAS,OAK,LAS,McCarran International,Las Vegas,NV,USA,36.080361,-115.152333
3,2001-01-01 01:39:00,4,256,LAS,PHX,LAS,McCarran International,Las Vegas,NV,USA,36.080361,-115.152333
4,2001-01-01 06:02:00,-6,377,MHT,BWI,MHT,Manchester,Manchester,NH,USA,42.934516,-71.437056


In [36]:
merged.isna().sum()

date           0
delay          0
distance       0
origin         0
destination    0
iata           0
name           0
city           0
state          0
country        0
latitude       0
longitude      0
dtype: int64

The missing city and state columns was irrelivant

In [39]:
merged[merged['iata'] == 'CLD']

,date,delay,distance,origin,destination,iata,name,city,state,country,latitude,longitude


In [41]:
print(merged.duplicated().sum())

0


Create 2 needed columns

In [45]:
merged['weekday'] = merged['date'].dt.weekday

In [ ]:
merged.head(20000)

,date,delay,distance,origin,destination,iata,name,city,state,country,latitude,longitude,weekday
0,2001-01-01 00:47:00,66,1750,DTW,LAS,DTW,Detroit Metropolitan-Wayne County,Detroit,MI,USA,42.212059,-83.348836,0
1,2001-01-01 01:10:00,95,2399,HNL,SFO,HNL,Honolulu International,Honolulu,HI,USA,21.318691,-157.922407,0
2,2001-01-01 01:24:00,-5,407,LAS,OAK,LAS,McCarran International,Las Vegas,NV,USA,36.080361,-115.152333,0
3,2001-01-01 01:39:00,4,256,LAS,PHX,LAS,McCarran International,Las Vegas,NV,USA,36.080361,-115.152333,0
4,2001-01-01 06:02:00,-6,377,MHT,BWI,MHT,Manchester,Manchester,NH,USA,42.934516,-71.437056,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2001-03-31 20:51:00,2,316,ORD,CLE,ORD,Chicago O'Hare International,Chicago,IL,USA,41.979595,-87.904464,5
19996,2001-03-31 21:09:00,5,73,DEN,COS,DEN,Denver Intl,Denver,CO,USA,39.858408,-104.667002,5
19997,2001-03-31 21:16:00,38,1426,MSP,PDX,MSP,Minneapolis-St Paul Intl,Minneapolis,MN,USA,44.880547,-93.216922,5
19998,2001-03-31 21:42:00,36,1172,DFW,IAD,DFW,Dallas-Fort Worth International,Dallas-Fort Worth,TX,USA,32.895951,-97.037200,5


In [53]:
merged['delayed_only'] = merged['delay'].where(merged['delay'] >= 0, 0)

In [54]:
merged.head()

,date,delay,distance,origin,destination,iata,name,city,state,country,latitude,longitude,weekday,delayed_only
0,2001-01-01 00:47:00,66,1750,DTW,LAS,DTW,Detroit Metropolitan-Wayne County,Detroit,MI,USA,42.212059,-83.348836,0,66
1,2001-01-01 01:10:00,95,2399,HNL,SFO,HNL,Honolulu International,Honolulu,HI,USA,21.318691,-157.922407,0,95
2,2001-01-01 01:24:00,-5,407,LAS,OAK,LAS,McCarran International,Las Vegas,NV,USA,36.080361,-115.152333,0,0
3,2001-01-01 01:39:00,4,256,LAS,PHX,LAS,McCarran International,Las Vegas,NV,USA,36.080361,-115.152333,0,4
4,2001-01-01 06:02:00,-6,377,MHT,BWI,MHT,Manchester,Manchester,NH,USA,42.934516,-71.437056,0,0


In [55]:
merged.shape

(20000, 14)

Delayed_only column added so one can see only the delayed ones adn analyse them, without the the early flights

Use group by to group to answer which airports, hours and weekdays have the worst delays.

In [65]:
merged.groupby('name')['delayed_only'].agg(['count', 'mean', 'median']).round(1).sort_values('mean', ascending=False).head(10)

,count,mean,median
name,,,
Central Illinois Regional,6,130.7,30.5
Ralph Wien Memorial,2,96.5,96.5
Bangor International,4,36.8,12.0
Dane County Regional,18,33.9,3.0
Monterey Peninsula,8,33.6,34.5
Juneau International,9,30.7,17.0
Killeen Municipal,13,28.4,0.0
Valley International,18,27.4,5.0
Daytona Beach International,11,27.2,0.0


In [66]:
merged.groupby('weekday')['delayed_only'].agg(['count', 'mean', 'median']).round(1).sort_values('mean', ascending=False).head(10)

,count,mean,median
weekday,,,
4,3003,15.5,3.0
3,2944,13.7,1.0
1,2903,12.4,0.0
6,2648,12.2,0.0
0,2952,12.1,0.0
2,2895,11.9,0.0
5,2655,10.2,0.0


Fridays followed by thursdays. A few large delays may have given larger mean, while the median is 0.0 for the third row.

Set a new column for hours

In [68]:
merged['hour'] = merged['date'].dt.hour

merged['hour']

0         0
1         1
2         1
3         1
4         6
         ..
19995    20
19996    21
19997    21
19998    21
19999    22
Name: hour, Length: 20000, dtype: int32

In [69]:
merged.groupby('hour')['delayed_only'].agg(['count', 'mean', 'median']).round(1).sort_values('mean', ascending=False).head(10)

,count,mean,median
hour,,,
3,2,121.0,121.0
2,9,63.9,12.0
0,70,35.0,10.0
23,171,31.1,5.0
1,53,30.5,5.0
22,456,26.9,5.0
21,711,23.4,6.0
19,1244,18.5,2.0
20,988,16.2,1.0


Small conclusion:
Flights with departure during the nights have more delays: Specially the two with higher number of counts, hours: 0, 23, 1, 22 and 21, since their counts are many.

Now another question, which period does the data cover?

In [74]:
period = merged['date'].max() - merged['date'].min()

print(merged['date'].max())
print(merged['date'].min())
print(period)

2001-03-31 22:27:00
2001-01-01 00:47:00
89 days 21:40:00


The data covers between 2001-01-01 to 2001-03-31, in total 89 days and 21 hours and 40 minutes.
It can help to predict well the data within that period, but outside is only reliable if we know the flights have been the same way. Otherwise it is not reliable.